***IMPORTS , importing used libraries***

In [ ]:
%pip install langchain langchain-community langchain-text-splitters langchain-huggingface langchain-core faiss-cpu fastapi uvicorn pyngrok nest-asyncio

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
import torch
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_core.runnables import RunnablePassthrough,RunnableLambda
import re

**Helper Functions**: Defines utility functions — `extract_json_block` extracts JSON from markdown code blocks, `add_line_numbers` adds line numbers to code for easier reference during review, and `review_python_file` reads a file and runs it through the AI review chain.

In [ ]:
#helper functions

def extract_json_block(text):
    pattern = r'```json\s*(.*?)\s*```'
    matches = re.findall(pattern, text, re.DOTALL)

    return f"```json\n{matches[-1]}\n```"


def add_line_numbers(code: str) -> str:
    return "\n".join(
        f"{i+1:4} | {line}"
        for i, line in enumerate(code.splitlines())
    )

def review_python_file(file_path: str):
    # Read the file
    with open(file_path, "r", encoding="utf-8") as f:
        code = f.read()

    # Add line numbers
    numbered_code = add_line_numbers(code)

    # Run the AI review
    response = chain.invoke(numbered_code)

    return response

***loading files from cheatsheets directory***

In [ ]:
cheatsheets_path = "/kaggle/input/datasets/yassinharraz/cheatsheet"
loader = DirectoryLoader(
    cheatsheets_path,
    glob="**/*.md",
    loader_cls=TextLoader
)

documents = loader.load()

In [ ]:
len(documents)

In [ ]:
#documents[77]

***now we will choose and implement chunking size and method***

In [ ]:
#creating tet splitter to split the documents into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=150
)
chunks = text_splitter.split_documents(documents)

In [ ]:
len(chunks)

In [ ]:
print(chunks[0].page_content)

***now we will embedd chunks and store in   FAISS***

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    encode_kwargs={
        "normalize_embeddings": True
    }
)

In [ ]:
## storing embedding in faais vector database
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)
type(vector_store)

In [ ]:
vector_store.index.ntotal

**implementing retrieval part**

In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)
#verfiying it works
query = "How should Python variables be named?"
results = retriever.invoke(query)
len(results)

In [ ]:
for i, doc in enumerate(results, start=1):
    print(f"Result {i}")
    print("=" * 60)
    print(doc.metadata["source"])
    print()
    print(doc.page_content)
    print("\n")

**Load LLM Model**: Installs bitsandbytes for 4-bit quantization, then loads the Qwen2.5-7B-Instruct model with 4-bit quantization to reduce memory usage. Wraps it in a HuggingFace pipeline and LangChain's HuggingFacePipeline for integration with the review chain.

In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
#Qwen/Qwen2.5-7B-Instruct
from transformers import BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-7B-Instruct"

# 1. Create the quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# 2. Load the tokenizer and the quantized model
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
text_generation_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=2048,
    temperature=0.2,
    do_sample=False,
    return_full_text=False,
)

llm = HuggingFacePipeline(
    pipeline=text_generation_pipeline
)

In [ ]:
response = llm.invoke(
    "Explain what SQL Injection is in two sentences."
)

print(response)

**Prompt & Output Schema**: Defines the system prompt that instructs the LLM to act as a Python code reviewer, the human prompt template with placeholders for context/code/format, a helper `format_docs` function, Pydantic models (`Issue` and `CodeReviewReport`) for structured output, and the `PydanticOutputParser` to parse LLM responses.

In [ ]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an expert Python code reviewer.

Your job is to review Python code using ONLY the provided documentation.

Review the code for:

- PEP8 violations
- Clean Code issues
- Security vulnerabilities
- Bugs
- Performance problems

For every issue you find:

- Explain why it is a problem.
- Suggest a fix.

If there are no issues, clearly state that no issues were found.

Return your response in a structured format.
"""
        ),
        (
            "human",
            """
Documentation:

{context}

----------------------------------------

Python Code:

{code}

----------------------------------------

Review this code.

Return your response using the following format:

{format_instructions}
"""
        ),
    ]
)

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
#output schema
from pydantic import BaseModel, Field

class Issue(BaseModel):
    severity: str = Field(
        description="Severity level: Low, Medium, or High"
    )

    line: int = Field(
        description="Line number where the issue occurs"
    )

    issue: str = Field(
        description="Short title of the detected issue"
    )

    explanation: str = Field(
        description="Detailed explanation of why this is a problem"
    )

    suggested_fix: str = Field(
        description="Recommended way to fix the issue"
    )


from typing import List

class CodeReviewReport(BaseModel):
    file: str = Field(
        description="Name of the analyzed Python file"
    )

    issues: List[Issue] = Field(
        description="List of detected issues"
    )

from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(
    pydantic_object=CodeReviewReport
) ##parser that  convert the LLM's output into a CodeReviewReport

**Build Review Chain**: Assembles the LangChain LCEL pipeline — retrieves relevant documentation context, passes the code and format instructions, sends everything through the prompt to the LLM, and parses the output into a structured `CodeReviewReport` object.

In [ ]:
chain = (
    {
        "context": retriever | format_docs,
        "code": RunnablePassthrough(),
        "format_instructions": RunnableLambda(
            lambda _: parser.get_format_instructions()
        ),
    }
    | prompt
    | llm
    | parser
    #| RunnableLambda(lambda x: x.model_dump_json(indent=2))    
)

In [ ]:
sample_code = """
def GetUser(ID):
    query = "SELECT * FROM users WHERE id=" + ID
    return query
"""

response = chain.invoke(sample_code)

print(response)
#print(response.model_dump_json(indent=2))


**Expose as REST API**: Creates a FastAPI endpoint at `/review` that accepts Python code via POST, runs it through the review chain, and returns the structured JSON report. Uses ngrok to create a public tunnel so the API is accessible from outside the Kaggle environment.

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import nest_asyncio
import uvicorn
from pyngrok import ngrok

# 1. Define the incoming request schema
class CodeRequest(BaseModel):
    code: str

# 2. Initialize the API
app = FastAPI()

# 3. Create the REST endpoint
@app.post("/review")
def review_code_endpoint(request: CodeRequest):
    try:
        # Pass the incoming code to your existing LangChain
        response = chain.invoke(request.code)
        
        # FastAPI automatically converts your CodeReviewReport 
        # Pydantic object into a clean JSON response
        return response
    except Exception as e:
        return {"error": str(e)}

# 4. Set up the Ngrok Tunnel
# WARNING: Keep your auth token private!
NGROK_AUTH_TOKEN = "3GxM8ZaaWPaPV2XOP7ClnWI4l6F_4U8dvR6nqngQnWnaf7w7Q"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Open a local tunnel on port 8000
ngrok_tunnel = ngrok.connect(8000)
public_url = ngrok_tunnel.public_url

print("=" * 60)
print(f"API is live!")
print(f"Public URL: {public_url}")
print(f"Send your POST requests to: {public_url}/review")
print("=" * 60)

# 5. Run the server inside the Kaggle notebook
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)

# Await the server to attach it to Kaggle's existing event loop
await server.serve()